In [1]:
# inference_sales_delta.py
 
import json, joblib
import pandas as pd
import numpy as np
 
MODEL_PATH   = "Models/sales_delta_rf_pipeline.joblib"
META_PATH    = "Models/sales_delta_rf_metadata.json"
COLUMNS_PATH = "Models/sales_delta_rf_columns.json"
 
def load_sales_delta_artifacts():
    pipe = joblib.load(MODEL_PATH)
    with open(META_PATH, "r") as f:
        meta = json.load(f)
    cols = pd.read_json(COLUMNS_PATH, typ="series").tolist()
    return pipe, meta, cols
 
def predict_sales_price_delta(df_new: pd.DataFrame) -> pd.Series:
    """
    df_new must include:
      - 'LATEST_PRICE' (ask price) — if values are in millions (e.g., 0.75), we convert to dollars.
      - Δ-feature columns (we reindex to training schema, filling missing with 0).
    Returns a Series 'SALE_PRICE_PRED'.
    """
    pipe, meta, cols = load_sales_delta_artifacts()
    assert "LATEST_PRICE" in df_new.columns, "LATEST_PRICE is required."
 
    df = df_new.copy()
 
    # Convert to dollars if looks like millions
    if df["LATEST_PRICE"].median() < 10:
        df["LATEST_PRICE"] = df["LATEST_PRICE"] * 1_000_000
 
    # Align to Δ-feature schema
    X_delta_new = df.reindex(columns=cols, fill_value=0)
 
    # Predict Δ and reconstruct SALE
    delta_hat = pipe.predict(X_delta_new)
    sale_hat  = df["LATEST_PRICE"].astype(float) + delta_hat
    return pd.Series(sale_hat, index=df.index, name="SALE_PRICE_PRED")